# Test Ingest (1 Day)
Pull 1 day from each data product, save to `cache/test_ingest_1day/`, then visualize from cache.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys, os

PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)
from libs import GOESData, HRRRData, NDVIData, OrographyData, LandMaskData

EXTENT = (-118.615, -117.70, 33.60, 34.35)
DIM = 84
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'test_ingest_1day')
os.makedirs(DATA_DIR, exist_ok=True)

START = '2024-08-01'
END   = '2024-08-02'

---
## Ingest

### GOES AOD / FRP / ADP

In [ ]:
goes_out = os.path.join(DATA_DIR, 'goes.npz')
if os.path.exists(goes_out):
    print('goes.npz: already cached, skipping')
else:
    goes_products = [
        ('aod',       'ABI-L2-AODC', 'AOD'),
        ('frp',       'ABI-L2-FDCC', 'Power'),
        ('adp_smoke', 'ABI-L2-ADPC', 'Smoke'),
        ('adp_dust',  'ABI-L2-ADPC', 'Dust'),
    ]
    goes_data = {}
    for name, product, subproduct in goes_products:
        print(f'Ingesting goes/{name}...')
        lib_cache = os.path.join(DATA_DIR, f'goes_{name}_lib.npz')
        g = GOESData(
            start_date=f'{START} 00:00',
            end_date=f'{END} 00:00',
            extent=EXTENT, dim=DIM,
            product=product, subproduct=subproduct,
            cache_path=lib_cache,
            load_cache=os.path.exists(lib_cache),
            save_cache=True,
            verbose=True,
        )
        goes_data[name] = g.data
        print(f'  {name}: {g.data.shape}\n')
    np.savez_compressed(goes_out, **goes_data)
    print(f'Saved goes.npz ({len(goes_data)} products)')

### HRRR

In [ ]:
hrrr_out = os.path.join(DATA_DIR, 'hrrr.npz')
if os.path.exists(hrrr_out):
    print('hrrr.npz: already cached, skipping')
else:
    print('Ingesting HRRR...')
    h = HRRRData(
        start_date=f'{START}-00',
        end_date=f'{END}-00',
        extent=EXTENT,
        grid_size=DIM,
        output_dir=os.path.join(DATA_DIR, 'hrrr_work'),
        force_reprocess=True,
        verbose=True,
    )
    hrrr_data = {}
    for var, arr in h.data.items():
        hrrr_data[var] = arr
        print(f'  {var}: {arr.shape}')
    np.savez_compressed(hrrr_out, **hrrr_data)
    print(f'Saved hrrr.npz ({len(hrrr_data)} variables)')

### MODIS NDVI

In [ ]:
ndvi_out = os.path.join(DATA_DIR, 'ndvi.npz')
if os.path.exists(ndvi_out):
    print('ndvi.npz: already cached, skipping')
else:
    print('Ingesting MODIS NDVI...')
    ndvi_work = os.path.join(DATA_DIR, 'ndvi_work')
    os.makedirs(ndvi_work, exist_ok=True)
    n = NDVIData(
        start_date=START,
        end_date='2024-08-17',
        extent=EXTENT,
        dim=DIM,
        raw_dir=ndvi_work,
        save_dir=ndvi_work,
    )
    np.savez_compressed(ndvi_out, ndvi=n.data)
    print(f'  Saved ndvi.npz: {n.data.shape}')

### Static Stubs

In [ ]:
static_out = os.path.join(DATA_DIR, 'static.npz')
if os.path.exists(static_out):
    print('static.npz: already cached, skipping')
else:
    T = 24
    static_data = {}
    for name, Cls in [('orography', OrographyData), ('land_mask', LandMaskData)]:
        stub = Cls(extent=EXTENT, dim=DIM)
        tiled = np.tile(stub.data, (T, 1, 1))
        static_data[name] = tiled
        print(f'{name}: {tiled.shape} (stub — all zeros)')
    np.savez_compressed(static_out, **static_data)
    print('Saved static.npz')

---
## Visualize from Cache

In [ ]:
def load_npz(npz_name):
    """Load a grouped .npz and print stats for each key."""
    path = os.path.join(DATA_DIR, f'{npz_name}.npz')
    data = np.load(path)
    result = {}
    for key in data:
        arr = data[key]
        nan_pct = np.isnan(arr).mean() * 100
        print(f'{npz_name}/{key}: shape={arr.shape}, NaN={nan_pct:.1f}%, '
              f'range=[{np.nanmin(arr):.4g}, {np.nanmax(arr):.4g}]')
        result[key] = arr
    return result

def show(data, titles, suptitle=None, cmap='viridis', t_idx=0):
    n = len(data)
    fig, axes = plt.subplots(1, n, figsize=(4 * n, 4))
    if n == 1: axes = [axes]
    for ax, arr, title in zip(axes, data, titles):
        frame = arr[t_idx] if arr.ndim == 3 else arr
        im = ax.imshow(frame, cmap=cmap)
        ax.set_title(title, fontsize=10)
        ax.axis('off')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    if suptitle:
        fig.suptitle(suptitle, fontsize=13)
    plt.tight_layout()
    plt.show()

### GOES

In [ ]:
goes = load_npz('goes')

t = min(20, len(goes['aod']) - 1)
show(
    [goes['aod'], goes['frp'], goes['adp_smoke'], goes['adp_dust']],
    ['AOD', 'FRP Power', 'ADP Smoke', 'ADP Dust'],
    suptitle=f'GOES  (frame {t})', t_idx=t
)

### HRRR

In [ ]:
hrrr = load_npz('hrrr')

show(
    [hrrr['temp_2m'], hrrr['dewpoint_2m'], hrrr['rh_2m'],
     hrrr['pressure_surface'], hrrr['pressure_msl']],
    ['Temp 2m', 'Dewpoint 2m', 'RH 2m', 'P Sfc', 'P MSL'],
    suptitle='HRRR Thermodynamic  (frame 12)', t_idx=12
)

In [ ]:
show(
    [hrrr['pbl_height'], hrrr['u_wind'], hrrr['v_wind'], hrrr['smoke_massden']],
    ['PBL Height', 'U Wind', 'V Wind', 'Smoke'],
    suptitle='HRRR Dynamics & Smoke  (frame 12)', t_idx=12
)

### MODIS NDVI

In [ ]:
ndvi_data = load_npz('ndvi')
show([ndvi_data['ndvi']], ['NDVI'], suptitle='MODIS NDVI', cmap='YlGn', t_idx=0)

### Static Stubs

In [ ]:
static = load_npz('static')
show(
    [static['orography'], static['land_mask']],
    ['Orography (stub)', 'Land Mask (stub)'],
    suptitle='Static Fields — zeros until implemented', cmap='terrain'
)